# M3 R1 robustness: does the model look at the snake or at the scenery?

No training. One trained run's `best.pt` (run this notebook once for F0 and once for the FE winner) rescores every
validation photo four ways (rulings R69-R71):

- **(a) as is**, the input the run was validated on;
- **(b) snake blanked**: the snake's mask pixels set to the ImageNet mean grey (0 after normalisation), every other
  pixel unchanged. The **shortcut rate** is the share of photos right as they are that stay right without the snake;
- **(c) background swapped**: the snake's pixels pasted, unchanged, on the scenery of a validation photo of another
  class (a seeded draw; the donor's own snake is blanked first, then the donor is resized to this photo). We report how
  often the prediction keeps its own class and how often it follows the donor's, against the control of how often it
  already named the donor's class before the swap. The **silhouette control** costs no GPU time: how often the donor's
  own blanked photo is called the donor's class, and follows-donor again without those swaps;
- **(d) slices**: Indian against non-Indian photos, dark (notebook 06's exposure rule), blurry (Laplacian variance
  below `BLUR_BELOW`), and classes with 10 or fewer training photographers (R89: 29 classes, 119 validation photos;
  R69's cut-off of 3 left the slice empty), each with photographer-grouped bootstrap intervals.

**Masks:** validation photos have no SAM masks, so (b) and (c) use the front end's **predicted** masks
(`frontend_preds.csv`; the segmenter that wrote them is recorded in the metrics' provenance, from the predictions'
`summary.json`). Photos without a usable predicted mask of the photo's size are left out of (b) and (c) and counted.

**Caveats** for the shortcut rate and follows-donor: a predicted mask can miss part of the snake (a tail tip, a
coil), and those pixels survive blanking, so both are upper bounds on reliance on scenery; and a blanked snake leaves a
grey silhouette with its outline, which a model can read as shape. The silhouette control above measures the second
for (c).

The verdict-level effect reads `decision.decide` with the thresholds `decision_report` fits on the run's own calibration
outputs, with the same `RECALL_BOUND` as the run's decision file (R104b). A by-product, `quality.csv` (path, set,
dark, blurry, brightness, Laplacian variance) for the validation,
calibration and unknown-pool half A photos, is the quality table `decision_report(quality=...)` reads (R37).

Code: `../src` (the runner kernel unpacks it into `/kaggle/working/repo`). Datasets through
`kagglehub.dataset_download`: `serpentguard-indian-snakes-clean` and `serpentguard-m3-subset`. The run and the
predictions come from other kernels' outputs through `kagglehub.notebook_output_download`, which a batch kernel can
read only when they are declared as kernel sources: `RUN_KERNEL`, the training kernel, whose output holds the folder
`RUN_ID` (`best.pt`, `config.yaml`, `calib_outputs.npz`, `prototypes.npz`, `val_outputs.npz`), and `PREDS_KERNEL`,
the notebook 13 kernel, whose output holds `frontend_preds.csv` and `masks_pred/`. Outputs:
`/kaggle/working/robustness/quality.csv` and `/kaggle/working/robustness/<RUN_ID>/` (`robustness_metrics.json` and
each condition's per-photo outputs, verdicts included, embeddings as float16). One GPU.

**Unattended runs (R91, R94):** the subset is checked before any GPU time; the as-is pass must reproduce the run's
own saved validation top-1 (`INSTRUMENT_FLOOR`) before (b) and (c) start; each condition's outputs are saved as its
pass ends. Running again on the same output folder keeps a complete `quality.csv` and every condition already saved
for the same photos, checkpoint, predictions, fitted thresholds, danger table, prototype set, fill and donor seed. A
photo that cannot be built is dropped and counted under `photos.errors`; a failing machine (out of disk or memory)
stops the run.

In [1]:
RUN_KERNEL = 'dataset:puneetbajajiitm/serpentguard-m3-run-final'
RUN_ID = 'H1_top1_s1'
PREDS_KERNEL = 'puneetbajajiitm/sg-m3-frontends'
FEW_PHOTOGRAPHERS = None  # None keeps R89's cut-off (10: 29 classes, 119 validation photos)
LIMIT = None              # first N photos of each set, for a smoke run
RECALL_BOUND = 'lower'

In [2]:
import os
import subprocess
import sys

ALBUMENTATIONS_VERSION = "2.0.8"  # requirements-m3.txt; the evaluation transform the runs were scored with
TIMM_VERSION = "1.0.31"            # requirements-m3.txt; the encoders' weights load into it
subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"albumentations=={ALBUMENTATIONS_VERSION}",
                f"timm=={TIMM_VERSION}"], check=True)
os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"  # otherwise albumentations asks PyPI for its latest version on import

import albumentations
import timm

assert albumentations.__version__ == ALBUMENTATIONS_VERSION, f"albumentations {albumentations.__version__} is loaded"
assert timm.__version__ == TIMM_VERSION, f"timm {timm.__version__} is loaded"
print(f"albumentations {albumentations.__version__}, timm {timm.__version__}")

albumentations 2.0.8, timm 1.0.31


In [3]:
SCIKIT_LEARN_VERSION = "1.9.1"  # requirements-m3.txt, a Global Constraint: every metric uses it
subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"scikit-learn=={SCIKIT_LEARN_VERSION}"], check=True)

import sklearn

assert sklearn.__version__ == SCIKIT_LEARN_VERSION, (f"scikit-learn {sklearn.__version__} is loaded; restart the "
                                                     f"kernel after the install")
print(f"scikit-learn {sklearn.__version__}")

scikit-learn 1.9.1


In [4]:
import json
import time
from pathlib import Path

import kagglehub

CODE = Path("/kaggle/input/serpentguard-code/src")
CODE = CODE if CODE.exists() else Path("../src").resolve()
sys.path.insert(0, str(CODE))

import pandas as pd
import torch

from serpentguard.config import load_config
from serpentguard.robustness import INSTRUMENT_FLOOR, METRICS_FILE, QUALITY_FILE
from serpentguard.run_data import FRONTEND_PREDS_FILE
from serpentguard.run_outputs import subset_hash
from serpentguard.train import (CALIBRATION_OUTPUTS, CHECKPOINT_FILE, CONFIG_FILE, HASH_FILE, PROTOTYPES_FILE,
                                VAL_OUTPUTS)

OWNER = "puneetbajajiitm"


def input_root(slug):
    """Where an attached dataset is mounted; Kaggle no longer mounts datasets at /kaggle/input/<slug>."""
    return Path(kagglehub.dataset_download(f"{OWNER}/{slug}"))


DATASET_PREFIX = "dataset:"  # "dataset:owner/slug": a private dataset copy of a kernel's output, for another account


def kernel_output(handle):
    """Where a kernel's output is: "owner/slug" reads it (a batch kernel only once it is declared a kernel source),
    "dataset:owner/slug" a private dataset copy of it."""
    if handle.startswith(DATASET_PREFIX):
        return Path(kagglehub.dataset_download(handle.removeprefix(DATASET_PREFIX)))
    return Path(kagglehub.notebook_output_download(handle))


DATA = input_root("serpentguard-indian-snakes-clean")
SUBSET = input_root("serpentguard-m3-subset") / "subset.csv"
RUN_OUTPUT = kernel_output(RUN_KERNEL)
runs = sorted(path.parent for path in RUN_OUTPUT.rglob(CHECKPOINT_FILE) if path.parent.name == RUN_ID)
assert len(runs) == 1, f"{RUN_KERNEL}'s output must hold one {RUN_ID} folder with {CHECKPOINT_FILE}; found {runs}"
RUN = runs[0]
preds_files = sorted(kernel_output(PREDS_KERNEL).rglob(FRONTEND_PREDS_FILE))
assert len(preds_files) == 1, f"{PREDS_KERNEL}'s output must hold one {FRONTEND_PREDS_FILE}; found {preds_files}"
PREDS = preds_files[0].parent
OUT = Path("/kaggle/working/robustness")
POLL_SECONDS = 60
EXPECTED_SUBSET_HASH = "40196d19a68e"  # first 12 hex digits of the real subset's SHA-256

for required in (DATA / "metadata.csv", DATA / "classes.csv", DATA / "data", SUBSET, PREDS / "masks_pred",
                 *(RUN / name for name in (CONFIG_FILE, HASH_FILE, CALIBRATION_OUTPUTS, PROTOTYPES_FILE, VAL_OUTPUTS))):
    assert required.exists(), f"missing input {required}"
cfg = load_config(RUN / CONFIG_FILE)
assert cfg.run_id == RUN_ID, f"{RUN} holds run {cfg.run_id}"
subset_digest = subset_hash(pd.read_csv(SUBSET, usecols=["path"])["path"])  # before any GPU time
assert subset_digest.startswith(EXPECTED_SUBSET_HASH), f"{SUBSET} is another subset ({subset_digest[:12]})"
assert (RUN / HASH_FILE).read_text().strip() == subset_digest, f"{RUN_ID} was trained on another subset"
assert torch.cuda.is_available(), "no GPU in this session"
workers = os.cpu_count() or 2
OUT.mkdir(parents=True, exist_ok=True)
ENV = {**os.environ, "PYTHONPATH": os.pathsep.join(filter(None, [str(CODE), os.environ.get("PYTHONPATH")]))}
SMOKE = [] if LIMIT is None else ["--limit", str(LIMIT)]
SLICE = [] if FEW_PHOTOGRAPHERS is None else ["--few-photographers", str(FEW_PHOTOGRAPHERS)]
print(f"code {CODE}; {torch.cuda.device_count()} GPUs, {workers} CPUs; run {RUN_ID} ({cfg.front_end}) at {RUN}; "
      f"predictions {PREDS}")

code /kaggle/working/repo/src; 2 GPUs, 4 CPUs; run H1_top1_s1 (F1) at /kaggle/input/datasets/puneetbajajiitm/serpentguard-m3-run-final/H1_top1_s1; predictions /kaggle/input/notebooks/puneetbajajiitm/sg-m3-frontends/frontend_preds


## 1. Page cache, then the quality table

A cold read on the dataset mount costs ~60 ms of latency; reading every photo and predicted mask once, 64 at a time,
puts them in the page cache that every process shares. Then one CPU process measures every validation, calibration and
unknown-pool half A photo as stored (before the exposure fix) into `quality.csv`.

In [5]:
from serpentguard.data import IMAGE_FOLDER
from serpentguard.robustness import quality_frames, read_inputs, validation_frame
from serpentguard.run_data import WARM_THREADS, warm_page_cache

tables = read_inputs(DATA, SUBSET, PREDS)
photos = [DATA / IMAGE_FOLDER / path for frame in quality_frames(tables.metadata, tables.subset).values()
          for path in frame["path"]]
masks = validation_frame(tables, cfg, DATA, PREDS).query("mask_valid")["mask_file"]
warmed = warm_page_cache([*photos, *masks], threads=WARM_THREADS)
print(f"page cache: {warmed['files']:,} files ({warmed['missing']} missing), {warmed['mb']:,.0f} MB in "
      f"{warmed['seconds']:.0f} s")

page cache: 41,578 files (0 missing), 3,569 MB in 128 s


In [6]:
def last_line(path):
    lines = path.read_text(errors="replace").strip().splitlines() if path.exists() else []
    return lines[-1][-200:] if lines else "(no output yet)"


def run_module(name, args, log_path):
    """Runs `python -m serpentguard.robustness` with args, streaming its output to log_path; fails loudly."""
    command = [sys.executable, "-u", "-m", "serpentguard.robustness", *args]
    print(f"{name}: {' '.join(command)}")
    started = time.time()
    with open(log_path, "w") as log:
        process = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT, env=ENV)
        while process.poll() is None:
            time.sleep(POLL_SECONDS)
            print(f"{(time.time() - started) / 60:5.1f} min  {name}: {last_line(log_path)}")
    if process.returncode:
        print(log_path.read_text(errors="replace")[-5000:])
    assert process.returncode == 0, f"{name} exited with {process.returncode}; its log tail is above"
    print(last_line(log_path))


run_module("quality", ["quality", "--data-root", str(DATA), "--subset", str(SUBSET), "--out", str(OUT), "--workers",
                       str(workers), *SMOKE], OUT / "quality.log")

quality: /usr/bin/python3 -u -m serpentguard.robustness quality --data-root /kaggle/input/datasets/puneetbajajiitm/serpentguard-indian-snakes-clean --subset /kaggle/input/datasets/puneetbajajiitm/serpentguard-m3-subset/subset.csv --out /kaggle/working/robustness --workers 4
  1.0 min  quality: (no output yet)
  2.0 min  quality: 4087225234680194}, 'calibration': {'photos': 7999, 'dark': 0.06350793849231154, 'blurry': 0.04963120390048756}, 'unknown': {'photos': 451, 'dark': 0.11529933481152993, 'blurry': 0.015521064301552107}}
4087225234680194}, 'calibration': {'photos': 7999, 'dark': 0.06350793849231154, 'blurry': 0.04963120390048756}, 'unknown': {'photos': 451, 'dark': 0.11529933481152993, 'blurry': 0.015521064301552107}}


## 2. Score the run on the GPU

The run's `robustness_metrics.json` records provenance (the run and predictions kernels, the checkpoint's and
`frontend_preds.csv`'s SHA-256, the segmenter), the photos with and without a usable predicted mask and those dropped
because they could not be built, an instrument check (the as-is pass against the run's own saved validation outputs,
photo by photo; below `INSTRUMENT_FLOOR` the run stops before (b) and (c)), the three conditions with their caveats,
the verdict changes, the slices, the quality shares, the settings, the page-cache warm-up, versions, the subset hash
and which conditions were read back from an earlier run.

In [7]:
RUN_OUT = OUT / RUN_ID
RUN_OUT.mkdir(parents=True, exist_ok=True)
run_module(RUN_ID, ["score", "--run-dir", str(RUN), "--data-root", str(DATA), "--subset", str(SUBSET), "--preds-root",
                    str(PREDS), "--quality", str(OUT / QUALITY_FILE), "--out", str(RUN_OUT), "--device", "cuda:0",
                    "--num-workers", str(workers), "--run-kernel", RUN_KERNEL, "--preds-kernel", PREDS_KERNEL, *SMOKE,
                    *SLICE, "--recall-bound", RECALL_BOUND], RUN_OUT / "robustness.log")

H1_top1_s1: /usr/bin/python3 -u -m serpentguard.robustness score --run-dir /kaggle/input/datasets/puneetbajajiitm/serpentguard-m3-run-final/H1_top1_s1 --data-root /kaggle/input/datasets/puneetbajajiitm/serpentguard-indian-snakes-clean --subset /kaggle/input/datasets/puneetbajajiitm/serpentguard-m3-subset/subset.csv --preds-root /kaggle/input/notebooks/puneetbajajiitm/sg-m3-frontends/frontend_preds --quality /kaggle/working/robustness/quality.csv --out /kaggle/working/robustness/H1_top1_s1 --device cuda:0 --num-workers 4 --run-kernel dataset:puneetbajajiitm/serpentguard-m3-run-final --preds-kernel puneetbajajiitm/sg-m3-frontends --recall-bound lower
  1.0 min  H1_top1_s1: (no output yet)
  2.0 min  H1_top1_s1: (no output yet)
  3.0 min  H1_top1_s1: (no output yet)
  4.0 min  H1_top1_s1: (no output yet)
  5.0 min  H1_top1_s1: (no output yet)
  6.0 min  H1_top1_s1: (no output yet)
  7.0 min  H1_top1_s1: unreadable': 0, 'blanked': 15977, 'swapped': 15977, 'without_donor': 0, 'errors': {'as

## 3. Results

Shortcut rate and donor-following are on the photos with a usable predicted mask; the control column is how
often the photo was already called the donor's class before the swap, and the silhouette rows are the control for the
donor's blanked snake. Both rates are upper bounds where masks miss part of the snake. Slice gaps are top-1 inside the
slice minus outside it.

In [8]:
def interval(values):
    return f"[{values[0]:.3f}, {values[1]:.3f}]"


quality = pd.read_csv(OUT / QUALITY_FILE)
assert quality["path"].is_unique and set(quality["set"]) == {"val", "calibration", "unknown"}, "quality.csv sets"
assert not quality["path"].str.contains(r"(?:^|/)test(?:_extra)?/").any(), "quality.csv holds a test photo"
metrics = json.loads((RUN_OUT / METRICS_FILE).read_text())
assert metrics["subset_hash"].startswith(EXPECTED_SUBSET_HASH), f"{RUN_ID} read another subset"
check = metrics["instrument_check"]
assert check["top1_agreement"] >= INSTRUMENT_FLOOR, f"{RUN_ID}: the as-is pass disagrees with the run: {check}"
conditions, verdicts, slices = metrics["conditions"], metrics["verdicts"], metrics["slices"]
shortcut, donor = conditions["blanked"]["shortcut"], conditions["swapped"]["donor"]
photos = metrics["photos"]
summary = pd.Series({
    "run kernel": RUN_KERNEL, "run": RUN_ID, "front end": metrics["front_end"],
    "segmenter": metrics["provenance"]["segmenter"], "val photos": metrics["photos"]["val"],
    "photos dropped (as is / blanked / swapped)": " / ".join(str(photos["errors"][name])
                                                         for name in ("as_is", "blanked", "swapped")),
    "with mask": photos["with_mask"], "without mask": photos["without_mask"],
    "masks dropped by cleaning / wrong size / unreadable": " / ".join(
        str(photos[key]) for key in ("mask_dropped_by_cleaning", "mask_size_mismatch", "mask_unreadable")),
    "top-1 as is": conditions["as_is"]["top1"], "macro-F1 as is": conditions["as_is"]["macro_f1"],
    "top-1 as is (masked)": conditions["as_is_masked"]["top1"], "top-1 blanked": conditions["blanked"]["top1"],
    "macro-F1 blanked": conditions["blanked"]["macro_f1"], "shortcut rate": shortcut["shortcut_rate"],
    "shortcut 95% CI": interval(shortcut["shortcut_rate_ci"]),
    "shortcut India / not": f"{conditions['blanked']['by_india']['india']['shortcut_rate']:.3f} / "
                            f"{conditions['blanked']['by_india']['not_india']['shortcut_rate']:.3f}",
    "top-1 swapped": conditions["swapped"]["top1"], "keeps own class": donor["keeps_own_class"],
    "follows donor": donor["follows_donor"], "donor class as is (control)": donor["follows_donor_as_is"],
    "donor silhouette rate": donor["donor_silhouette_rate"],
    "follows donor without silhouette": donor["follows_donor_without_silhouette"],
    "verdict changed, blanked": verdicts["blanked"]["changed"],
    "dangerous -> not listed, blanked": verdicts["blanked"]["transitions"]["dangerous->not_listed"],
    "unknown when blanked": verdicts["blanked"]["after"]["unknown"] / verdicts["blanked"]["photos"],
    "verdict changed, swapped": verdicts["swapped"]["changed"],
    **{f"{name} gap": slices[name]["top1_gap"] for name in slices},
    **{f"{name} photos in / out": f"{slices[name]['in']['photos']} / {slices[name]['out']['photos']}"
       for name in slices},
    "few-photographer classes (cut-off)": f"{metrics['settings']['few_photographer_classes']} "
                                          f"(<= {metrics['settings']['few_photographers_at_most']})",
    "India gap 95% CI": interval(slices["india"]["top1_gap_ci"]),
    "instrument top-1 agreement": check["top1_agreement"], "read back": ", ".join(metrics["resumed"]) or "none",
    "minutes": metrics["seconds"] / 60})
summary.to_csv(RUN_OUT / "summary.csv", header=False)
print("quality flags (share of photos):")
print(quality.groupby("set")[["dark", "blurry"]].mean().round(3).to_string())
summary

quality flags (share of photos):
              dark  blurry
set                       
calibration  0.064   0.050
unknown      0.115   0.016
val          0.134   0.041


run kernel                                             dataset:puneetbajajiitm/serpentguard-m3-run-final
run                                                                                           H1_top1_s1
front end                                                                                             F1
segmenter                                              {'candidate': 'unet', 'kind': 'unet', 'weights...
val photos                                                                                         17151
photos dropped (as is / blanked / swapped)                                                     0 / 0 / 0
with mask                                                                                          15977
without mask                                                                                        1174
masks dropped by cleaning / wrong size / unreadable                                            0 / 0 / 0
top-1 as is                                            